In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import keras.ops as K
from keras.layers import Input, Flatten, Dense

# from keras.models import Sequential
from deel.lip.model import Sequential

from deel.lip.layers import (
    SpectralDense,
    SpectralConv2D,
    ScaledL2NormPooling2D,
    FrobeniusDense,
)
from deel.lip.activations import GroupSort, GroupSort2
from deel.lip.losses import HKR, KR, HingeMargin, MulticlassHKR, MulticlassKR

import numpy as np
import decomon

import sys

# setting path
sys.path.append('..')

from data_processing import load_data, select_data_for_radius_evaluation
from radius_evaluation_tools import compute_binary_certificate, starting_point_dichotomy
from lipschitz_decomon_tools import echantillonner_boule_l2_simple
from LearnedHyperplaneTools import *

In [2]:
x_train, x_test, y_train, y_test, y_test_ord = load_data("MNIST")

In [3]:
vanilla_model = keras.models.load_model("/home/aws_install/robustess_project/lip_models/demo0_vanilla_MNIST_channelfirst_False_disj_Neurons.keras")
vanilla_model.compile(
        # decreasing alpha and increasing min_margin improve robustness (at the cost of accuracy)
        # note also in the case of lipschitz networks, more robustness require more parameters.
        loss=MulticlassHKR(alpha=50, min_margin=0.05),
        optimizer=keras.optimizers.Adam(1e-3),
        metrics=["accuracy", MulticlassKR()],)

In [4]:
pt_choosen = 0

In [5]:
images, labels, idx_list = select_data_for_radius_evaluation(x_test, y_test_ord, vanilla_model)

In [6]:
x = images[pt_choosen:pt_choosen+1].flatten().detach().cpu().numpy()

In [7]:
vanilla_model(images[pt_choosen:pt_choosen+1])

tensor([[ 1.8283, -4.2363, -1.1456, -1.0565, -2.2833, -0.8807, -0.2685, -1.8158,
         -1.0744, -0.5852]], device='cuda:0', grad_fn=<MmBackward0>)

In [8]:
eps=2.5

In [9]:
y_list = [x]
for i in range(100):
    y_list.append(echantillonner_boule_l2_simple(x, eps))

In [10]:
from keras import layers

In [39]:
def create_difference_model(base_model, label, i):
    """
    Crée et retourne un nouveau modèle Keras qui calcule la différence
    entre le logit du 'label' et le logit de 'i'.
    """
    entree_base = base_model.inputs
    sortie_logits_base = base_model.outputs[0]
    # print(label)
    # Définition de la couche Lambda avec la correction et un nom unique
    difference = layers.Lambda(
        # lambda x, current_i=i: x[:, label] - x[:, current_i],
        lambda z: z[:, 0:1] - z[:, 1:2], 
        output_shape=(1,),
        # Nom de couche unique : très important !
        name=f"difference_{label}_vs_{i}"
    )(sortie_logits_base)

    # Création du modèle avec un nom unique
    difference_model = keras.Model(
        inputs=entree_base,
        outputs=difference,
        name=f"model_diff_{label}_vs_{i}"
    )
    
    return difference_model

In [40]:
def get_local_maximum_multiclass_Learned_Hyperplane(x, label, eps, y_list, model, L=1):
    """
    Adaptation du getlocalmaximum au cas multiclasse. On vient borner fgt - fi qui est une fonction racine de 2 lip
    """
    n_classes = model.output_shape[-1]
    list_outputs = list(range(n_classes))
    # print(list_outputs)
    list_outputs.remove(label)
    # print(list_outputs)
    current_min = 1000
    for i in list_outputs:
        print(i)
        difference_model = create_difference_model(model, label, i)

        _, max_one_vs_all =  get_local_maximum_Learned_Hyperplane(x, 1, eps, y_list, difference_model, L=np.sqrt(2)*L)   
        if max_one_vs_all < current_min:
            current_min = max_one_vs_all
    return current_min

In [41]:
min = get_local_maximum_multiclass_Learned_Hyperplane(x, labels[pt_choosen], eps, y_list, vanilla_model)

1


/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer']
Received: inputs=Tensor(shape=(1, 1, 28, 28))
  warnings.warn(msg)


2
3
4
5
6
7
8
9


In [42]:
min

0.7306967